# STEP 2: Text Preprocessing Pipeline

In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import joblib
from sklearn.preprocessing import LabelEncoder

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\XPRISTO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\XPRISTO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\XPRISTO\AppData\Roaming\nltk_data...


In [2]:
# LOAD DATA
df = pd.read_csv(r'C:\Users\XPRISTO\Desktop\NLP\cyberbullying_eda.csv')
print(f"Loaded {len(df)} rows")

Loaded 47692 rows


In [3]:
# PREPROCESSING FUNCTIONS
lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

def clean_tweet(text):
    """Full preprocessing pipeline for a tweet."""
    text = str(text).lower()                              # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)            # remove URLs
    text = re.sub(r'@\w+', '', text)                      # remove @mentions
    text = re.sub(r'#(\w+)', r'\1', text)                 # keep hashtag text
    text = re.sub(r'[^\w\s]', '', text)                   # remove punctuation
    text = re.sub(r'\d+', '', text)                       # remove numbers
    text = re.sub(r'\s+', ' ', text).strip()              # collapse whitespace

    # Tokenize → remove stopwords → lemmatize
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]

    return ' '.join(tokens)

In [4]:
# APPLY PREPROCESSING
print("Preprocessing tweets...")
df['clean_text'] = df['tweet_text'].apply(clean_tweet)

Preprocessing tweets...


In [5]:
# Remove empty rows after cleaning
df = df[df['clean_text'].str.strip() != '']
print(f"Rows after cleaning: {len(df)}")

Rows after cleaning: 47249


In [6]:
# LABEL ENCODING
le = LabelEncoder()
df['label'] = le.fit_transform(df['cyberbullying_type'])

print("\nLabel Mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {i} → {cls}")


Label Mapping:
  0 → age
  1 → ethnicity
  2 → gender
  3 → not_cyberbullying
  4 → other_cyberbullying
  5 → religion


In [7]:
# Save encoder for inference
joblib.dump(le, r'C:\Users\XPRISTO\Desktop\NLP\label_encoder.pkl')
print("\nSaved: label_encoder.pkl")


Saved: label_encoder.pkl


In [8]:
# TRAIN / VAL / TEST SPLIT
from sklearn.model_selection import train_test_split

X = df['clean_text']
y = df['label']

# 70% train | 15% val | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"\nSplit sizes:")
print(f"  Train : {len(X_train)} ({len(X_train)/len(df)*100:.1f}%)")
print(f"  Val   : {len(X_val)}   ({len(X_val)/len(df)*100:.1f}%)")
print(f"  Test  : {len(X_test)}  ({len(X_test)/len(df)*100:.1f}%)")


Split sizes:
  Train : 33074 (70.0%)
  Val   : 7087   (15.0%)
  Test  : 7088  (15.0%)


In [9]:
# Save splits
train_df = pd.DataFrame({'clean_text': X_train, 'label': y_train,
                          'cyberbullying_type': df.loc[X_train.index, 'cyberbullying_type']})
val_df   = pd.DataFrame({'clean_text': X_val,   'label': y_val,
                          'cyberbullying_type': df.loc[X_val.index,   'cyberbullying_type']})
test_df  = pd.DataFrame({'clean_text': X_test,  'label': y_test,
                          'cyberbullying_type': df.loc[X_test.index,  'cyberbullying_type']})

train_df.to_csv(r'C:\Users\XPRISTO\Desktop\NLP\train.csv', index=False)
val_df.to_csv(r'C:\Users\XPRISTO\Desktop\NLP\val.csv',     index=False)
test_df.to_csv(r'C:\Users\XPRISTO\Desktop\NLP\test.csv',   index=False)

print("\n Preprocessing complete.")
print("   Saved: train.csv | val.csv | test.csv")


 Preprocessing complete.
   Saved: train.csv | val.csv | test.csv
